# Bài 10 · Làm sạch dữ liệu có cấu trúc

**Lập trình xử lý dữ liệu (LTXLDL) · 2627-1 · Viện TTNT, UET-VNU**

> 💡 File → **Save a copy in Drive** trước khi sửa.

**Mục tiêu buổi học** — sau notebook này, bạn:

1. Chẩn đoán ba họ lỗi: **thiếu – lệch – thừa**, và hiểu vì sao *cơ chế thiếu* quyết định cách xử.
2. Xử lý theo nguyên tắc **gắn cờ, đừng xoá**: điền theo nhóm + cắm cờ; ngưỡng outlier chọn
   sau khi *nhìn phân phối*.
3. Đóng gói thành **bộ quy tắc QA** + `qa_report` + so KPI trước/sau — đúng chuẩn bài tập lớn.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

URL = ("https://data.insideairbnb.com/chile/rm/santiago/"
       "2026-06-29/data/listings.csv.gz")
df = pd.read_csv(URL, usecols=[
    "id", "name", "neighbourhood_cleansed", "room_type", "price",
    "minimum_nights", "latitude", "longitude", "bedrooms",
    "review_scores_rating", "availability_365",
])
df["price_num"] = (df["price"].str.replace("$", "", regex=False)
                              .str.replace(",", "", regex=False).astype(float))
df.shape

## 1. Thiếu — đo, hỏi cơ chế, rồi mới xử

In [ ]:
# Đo mức thiếu
(df.isna().mean() * 100).round(1).sort_values(ascending=False).head(5)

Ba cột thiếu, ba **cơ chế** khác nhau:

| Cột | Vì sao trống? | Hệ quả |
|---|---|---|
| `review_scores_rating` (17,8%) | phòng **chưa có review** — thiếu có hệ thống | điền 0 hay mean đều sai nghĩa; phân tích điểm chỉ trên phòng có review, nói rõ |
| `bedrooms` (12,6%) | chủ nhà lười khai — gần ngẫu nhiên | điền theo nhóm + cắm cờ |
| `price_num` (4,6%) | có thể phòng ngừng cho thuê — thiếu mang thông tin | loại khỏi KPI giá, ghi rõ |

In [ ]:
# Kiểm tra nhanh: nhóm thiếu giá có "lệch" về đâu không?
df.groupby(df["price_num"].isna())["room_type"].value_counts(normalize=True).round(3)

In [ ]:
# Combo tử tế: cờ TRƯỚC, điền theo nhóm SAU
df["bedrooms_isna"] = df["bedrooms"].isna()
df["bedrooms"] = df["bedrooms"].fillna(
    df.groupby("room_type")["bedrooms"].transform("median"))

print("Còn thiếu:", df["bedrooms"].isna().sum(),
      "| đã điền:", df["bedrooms_isna"].sum(), "dòng (có cờ làm chứng)")

## 2. Lệch — sai miền trước, thống kê sau

In [ ]:
# Luật miền: những điều KHÔNG THỂ đúng
mien = {
    "gia_thieu": df["price_num"].isna(),
    "gia_khong_duong": df["price_num"] <= 0,
    "dem_toi_thieu_>365": df["minimum_nights"] > 365,
    "toa_do_ngoai_bien": ~(df["latitude"].between(-34, -33)
                           & df["longitude"].between(-71, -70)),
}
{k: int(v.sum()) for k, v in mien.items()}

In [ ]:
# Nhìn phân phối TRƯỚC khi chọn ngưỡng outlier
gia = df.loc[df["price_num"] > 0, "price_num"]

fig, axes = plt.subplots(1, 2, figsize=(10, 3.4))
axes[0].hist(gia, bins=60, color="#1E93AB")
axes[0].set_title("Thang thường: mù")
axes[1].hist(np.log10(gia), bins=60, color="#E8890C")
axes[1].set_title("Thang log10: sáng")
plt.tight_layout(); plt.show()

In [ ]:
# IQR cổ điển trên phân phối lệch phải -> gắn cờ quá tay
q1, q3 = gia.quantile([0.25, 0.75])
fence_iqr = q3 + 1.5 * (q3 - q1)
print(f"Fence IQR = {fence_iqr:,.0f} -> gắn cờ {(gia > fence_iqr).mean():.1%} thị trường (quá tay!)")

# Ngưỡng percentile: khiêm tốn và giải thích được
p99 = gia.quantile(0.99)
print(f"P99       = {p99:,.0f} -> gắn cờ {(gia > p99).mean():.1%}")

In [ ]:
# Soi tay vài "nghi phạm" trước khi kết án
df[df["price_num"] > p99].nlargest(3, "price_num")[
    ["name", "room_type", "review_scores_rating", "price_num"]]

## 3. Thừa — trùng lặp phụ thuộc vào KHOÁ

In [ ]:
print("Dòng trùng nguyên vẹn:", df.duplicated().sum())
print("id trùng trong snapshot:", df["id"].duplicated().sum())

In [ ]:
# Ghép 2 snapshot: id "trùng" hàng loạt — nhưng đó là dữ liệu panel, không phải lỗi!
t9 = pd.read_csv("https://data.insideairbnb.com/chile/rm/santiago/"
                 "2025-09-27/visualisations/listings.csv", usecols=["id"])
t9["snapshot"], nay = "2025-09", df[["id"]].assign(snapshot="2026-06")
ca_hai = pd.concat([t9, nay])

print("id trùng khi khoá = id          :", ca_hai["id"].duplicated().sum())
print("trùng khi khoá = (id, snapshot) :", ca_hai.duplicated().sum())

12.429 id xuất hiện ở cả hai snapshot = cùng một phòng, hai thời điểm — chính là thứ cho phép
phân tích "theo thời gian" của BTL. **Khoá của bảng gộp là `(id, snapshot)`**.

## 4. Đóng gói: bộ quy tắc QA + báo cáo tác động

In [ ]:
def qa_rules(d):
    """Mỗi quy tắc một mặt nạ. Chỉ PHÁT HIỆN — không sửa gì."""
    gia = d["price_num"]
    rules = {
        "gia_thieu":     gia.isna(),
        "gia_p99":       gia > gia.quantile(0.99),
        "dem_999":       d["minimum_nights"] > 365,
        "toa_do_lac":    ~(d["latitude"].between(-34, -33)
                           & d["longitude"].between(-71, -70)),
        "id_trung":      d["id"].duplicated(keep=False),
        "bedrooms_dien": d["bedrooms_isna"],
    }
    return rules

report = pd.DataFrame([
    {"quy_tac": ten, "so_dong": int(m.sum()), "ty_le_%": round(m.mean() * 100, 2)}
    for ten, m in qa_rules(df).items()
])
report

In [ ]:
# Gắn cờ tổng hợp + so KPI trước/sau
masks = qa_rules(df)
df["flag_gia"] = masks["gia_thieu"] | masks["gia_p99"]

truoc = df["price_num"].agg(["mean", "median"])
sau = df.loc[~df["flag_gia"], "price_num"].agg(["mean", "median"])
pd.DataFrame({"trước QA": truoc, "sau QA": sau}).round(0)

Chưa đến 1% số dòng bị cờ giá mà **mean phồng lên gần 44%** — median gần như bất động.
Bảng "trước/sau" này là bằng chứng thuyết phục nhất rằng QA của bạn có giá trị
(và là nội dung bắt buộc của báo cáo BTL).

## 5. Bài tập tại lớp

### Bài 1 — Thêm 2 quy tắc mới

Bổ sung vào `qa_rules`: (a) `diem_ngoai_thang` — `review_scores_rating` ngoài [0, 5];
(b) `ten_trong` — cột `name` rỗng hoặc chỉ toàn khoảng trắng. Chạy lại `report`.

In [ ]:
# TODO Bài 1:
def qa_rules_v2(d):
    r = qa_rules(d)
    r["diem_ngoai_thang"] = ~d["review_scores_rating"].between(0, 5) & d["review_scores_rating"].notna()
    r["ten_trong"] = d["name"].isna() | (d["name"].str.strip().str.len() == 0)
    return r

pd.DataFrame([{"quy_tac": t, "so_dong": int(m.sum())} for t, m in qa_rules_v2(df).items()])

### Bài 2 — Outlier nội quận (transform tái xuất)

Quy tắc `gia_p99` dùng ngưỡng **toàn thành phố** — một phòng 200.000 CLP là bình thường ở
Vitacura nhưng bất thường ở Cerro Navia. Viết quy tắc `gia_20x_quan`: giá gấp hơn 20 lần
**trung vị quận của nó**. Bao nhiêu dòng bị cờ? So với `gia_p99` — hai quy tắc bắt trùng nhau
bao nhiêu dòng?

In [ ]:
# TODO Bài 2:
he_so = df["price_num"] / df.groupby("neighbourhood_cleansed")["price_num"].transform("median")
gia_20x = he_so > 20
print("gia_20x_quan bắt:", int(gia_20x.sum()))
print("giao với gia_p99:", int((gia_20x & masks["gia_p99"]).sum()))

### Bài 3 — Bộ QA sống sót qua snapshot khác

Chạy `qa_rules` trên snapshot 09/2025 (bản visualisations — **schema khác**: giá là số sẵn,
không có `bedrooms`). Sửa `qa_rules` thành phiên bản *phòng thủ*: chỉ chạy quy tắc khi cột
cần thiết tồn tại. Đây chính là yêu cầu "chịu được schema drift" khi chấm held-out.

In [ ]:
# TODO Bài 3 (scaffold):
t9_full = pd.read_csv("https://data.insideairbnb.com/chile/rm/santiago/"
                      "2025-09-27/visualisations/listings.csv")
t9_full["price_num"] = t9_full["price"]      # bản này giá là số sẵn

def qa_rules_defensive(d):
    rules = {}
    if "price_num" in d:
        rules["gia_thieu"] = d["price_num"].isna()
        rules["gia_p99"] = d["price_num"] > d["price_num"].quantile(0.99)
    if "minimum_nights" in d:
        rules["dem_999"] = d["minimum_nights"] > 365
    if "id" in d:
        rules["id_trung"] = d["id"].duplicated(keep=False)
    return rules

pd.DataFrame([{"quy_tac": t, "so_dong": int(m.sum())}
              for t, m in qa_rules_defensive(t9_full).items()])

## 6. Thử thách về nhà 🏆 — Bộ QA cho thành phố của nhóm

Lấy thành phố nhóm bạn (dự kiến) nhận cho BTL:

1. Tải `listings.csv.gz` snapshot mới nhất; chạy chẩn đoán 3 họ lỗi như notebook này.
2. Đề xuất **≥8 quy tắc QA** theo bảng 4 phần (tên – điều kiện – lý do – hành động) —
   trong đó ít nhất 2 quy tắc *đặc thù cho thành phố đó* (gợi ý: biên toạ độ, tiền tệ,
   mùa vụ, quy định địa phương về giấy phép…).
3. Sinh `qa_report.csv` + bảng KPI giá trước/sau.
4. Ghi lại: quy tắc nào bạn tranh cãi nội bộ nhiều nhất? Vì sao chốt như hiện tại?

Sản phẩm này dùng được **nguyên vẹn** cho mốc "bản đề xuất tuần 8" của BTL.

In [ ]:
RUN_CHALLENGE = False

if RUN_CHALLENGE:
    CITY_URL = "https://data.insideairbnb.com/.../listings.csv.gz"   # đổi theo nhóm
    ...

---

## Tóm tắt buổi học

| Ý chốt | Vì sao quan trọng |
|---|---|
| Gắn cờ > xoá; quy tắc đủ 4 phần | Đảo ngược được, bảo vệ được khi vấn đáp |
| Cơ chế thiếu quyết định cách xử; điền theo nhóm + cắm cờ | Không bóp méo phân phối trong im lặng |
| Nhìn phân phối (log) rồi mới chọn ngưỡng; IQR quá tay với dữ liệu lệch | Ngưỡng có lý do, không phải phản xạ |
| Trùng lặp phụ thuộc khoá; panel = (id, snapshot) | Nền của phân tích đa snapshot BTL |
| qa_report + KPI trước/sau | Bằng chứng QA đáng tiền |

**Buổi sau:** dữ liệu *phi* cấu trúc — khi quy tắc bó tay, LLM vào việc. Nhớ mang API key
(hướng dẫn lấy key miễn phí ở đầu notebook Bài 11).